In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from Bio.PDB import PDBParser
import os
import requests

ModuleNotFoundError: No module named 'response'

In [3]:
sel = pd.read_csv("pdb_selection_fixed2.csv")
print(sel.head())

  pdb_code phase                                                gro  \
0     2bem     2  /work001/misc/bekker/kakC/dynamicsdb/raw/2/2be...   
1     5s2e     4  /work001/misc/bekker/kakC/dynamicsdb/raw/4/5s2...   
2     1ncx    4b  /work001/misc/bekker/kakC/dynamicsdb/raw/4b/1n...   
3     7p8f    4f  /work001/misc/bekker/kakC/dynamicsdb/raw/4f/7p...   
4     1l0c     4  /work001/misc/bekker/kakC/dynamicsdb/raw/4/1l0...   

   protein_atoms  protein_heavy_atoms  n_ca  n_residues  n_chains  \
0           2588                 1329   170         170         1   
1           2512                 1252   165         166         1   
2           2465                 1271   162         162         1   
3           2629                 1305   163         165         1   
4           2598                 1316   166         168         1   

   total_atoms     bin_edges  size_bin_id     bin_label  
0        17150  [1200, 1400)            6  [1200, 1400)  
1        21055  [1200, 1400)            6 

In [9]:
done = pd.read_csv("done_boxpred2_pdbs.txt", header=None, names=["pdb"])
sel = pd.read_excel("pdb_selection_fixed2.xlsx")

done["pdb"] = done["pdb"].astype(str).str.strip().str.lower()
sel["pdb_code"] = sel["pdb_code"].astype(str).str.strip().str.lower()

sel[sel["pdb_code"].isin(done["pdb"])]["pdb_code"].drop_duplicates().to_csv(
    "selected_done_pdbs.txt", index=False, header=False
)

In [11]:
#get the pdbs that are not in both lists

selected_pdb_list = pd.read_csv("selected_done_pdbs.txt", header=None, names=["pdb"])
pdb_list = pd.read_excel("pdb_selection_fixed2.xlsx")["pdb_code"].drop_duplicates().str.strip().str.lower()

not_in_both = set(pdb_list) - set(selected_pdb_list["pdb"])

print(f"PDBs in selection but not in done list: {len(not_in_both)}")
list(not_in_both)[:10]

PDBs in selection but not in done list: 6


['3b9c', '4k0y', '1wy9', '4dme', '1fe5', '6typ']

In [6]:
#see which directories are actually on the selection list

base_dir = Path("results/")
pdb_dirs = [set(p.name for p in base_dir.iterdir() if p.is_dir())]

pdb_set = set(p.lower().strip() for p in pdb_list)

common = sorted(pdb_set & pdb_dirs[0])

print(f"{len(common)} matches found")
print(common[:10])

100 matches found
['1a7u', '1dxx', '1fe5', '1fof', '1ggp', '1hia', '1kap', '1l0c', '1m2r', '1nbq']


In [7]:
#get the difference between the selection list and the directories

missing_dirs = sorted(pdb_set - pdb_dirs[0])
extra_dirs = sorted(pdb_dirs[0] - pdb_set)

print(f"{len(missing_dirs)} missing directories")
print(f"{len(extra_dirs)} extra directories")

0 missing directories
22 extra directories


In [14]:
import pandas as pd
import requests

# Load unique PDB IDs
pdb_ids = (
    pd.read_excel("pdb_selection_fixed2.xlsx")["pdb_code"]
    .dropna()
    .drop_duplicates()
    .astype(str)
    .str.strip()
    .str.lower()
)

results = []

for pdb_id in pdb_ids:
    url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}"

    try:
        response = requests.get(url)

        if response.status_code != 200:
            results.append({
                "PDB": pdb_id,
                "State": "Not found",
                "Chains": None,
                "Assemblies": None,
                "Polymer entities": None
            })
            continue

        data = response.json()

        entry_info = data.get("rcsb_entry_info", {})

        assemblies = entry_info.get("assembly_count", None)
        polymer_entities = entry_info.get("polymer_entity_count", None)

        deposited_chains = entry_info.get(
            "deposited_polymer_entity_instance_count",
            None
        )

        if deposited_chains == 1:
            state = "Monomer"
        elif deposited_chains is None:
            state = "Unknown"
        else:
            state = f"Multimer ({deposited_chains} chains)"

        results.append({
            "PDB": pdb_id,
            "State": state,
            "Chains": deposited_chains,
            "Assemblies": assemblies,
            "Polymer entities": polymer_entities
        })

    except Exception as e:
        results.append({
            "PDB": pdb_id,
            "State": f"Error: {e}",
            "Chains": None,
            "Assemblies": None,
            "Polymer entities": None
        })

df = pd.DataFrame(results)

df.to_csv("results/000analysis/pdb_monomer_multimer_status.csv", index=False)

In [ ]:
#check pure monomer/multimer count

summary = df["State"].value_counts()
print(summary)


State
Monomer                 38
Multimer (2 chains)     33
Multimer (3 chains)     12
Multimer (4 chains)      8
Multimer (6 chains)      5
Multimer (8 chains)      2
Multimer (10 chains)     1
Multimer (24 chains)     1
Name: count, dtype: int64
